# Base de Datos Vectorial: Pinecone
## Motor de búsqueda semántica de ofertas laborales

**Proyecto Integrador 1 — Ingeniería de Sistemas**

### Objetivo del notebook

Reemplazar el índice FAISS en memoria (usado en `03_Embeddings_BusquedaVectorial.ipynb`) por una base de datos vectorial persistente: **Pinecone**, en su capa gratuita.

Decisiones tomadas para esta primera iteración:
- **Qué se indexa**: solo las **3,760 plantillas únicas** de texto (Opción A) — igual que con FAISS, no cada una de las 1.6M ofertas individuales.
- **Por qué Pinecone primero**: es la opción más simple de poner en marcha (servicio administrado, sin instalar/mantener infraestructura), útil para validar rápido cómo se siente trabajar con una base de datos vectorial real antes de decidir si migrar a algo self-hosted (Qdrant/Chroma) para el despliegue final en el VPS.

Este notebook asume que ya ejecutaste `02_Preprocesamiento.ipynb` y `03_Embeddings_BusquedaVectorial.ipynb` al menos una vez, de forma que en Google Drive (`proyecto_integrador/data/processed/`) existan:
- `job_descriptions_clean.parquet` (dataset limpio, sin `template_id`)
- `job_id_template_map.parquet` (mapeo `Job Id -> template_id`)
- `plantillas_meta.parquet` (una fila por plantilla única, con su `texto_combinado`)
- `embeddings_plantillas.npy` (embeddings ya generados de esas plantillas)

## 1. Preparación e instalación

In [ ]:
# !pip install -q pinecone sentence-transformers


In [ ]:
import time

import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


## 2. Cargar plantillas, embeddings y dataset limpio desde Drive

No hace falta volver a generar embeddings — ya se calcularon una vez en el notebook 03 y se guardaron en Drive. Aquí solo se leen.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
BASE_DIR = Path("/content/drive/MyDrive/proyecto_integrador")
RUTA_PROCESSED = BASE_DIR / "data" / "processed"

archivos_requeridos = [
    "job_descriptions_clean.parquet",
    "job_id_template_map.parquet",
    "plantillas_meta.parquet",
    "embeddings_plantillas.npy",
]
faltantes = [f for f in archivos_requeridos if not (RUTA_PROCESSED / f).exists()]
if faltantes:
    raise FileNotFoundError(
        f"Faltan estos archivos en {RUTA_PROCESSED}: {faltantes}. "
        "Ejecuta primero 02_Preprocesamiento.ipynb y 03_Embeddings_BusquedaVectorial.ipynb."
    )

print("Todos los archivos requeridos están disponibles.")


In [ ]:
plantillas = pd.read_parquet(RUTA_PROCESSED / "plantillas_meta.parquet")
embeddings = np.load(RUTA_PROCESSED / "embeddings_plantillas.npy")

df_clean = pd.read_parquet(RUTA_PROCESSED / "job_descriptions_clean.parquet")
mapeo_template = pd.read_parquet(RUTA_PROCESSED / "job_id_template_map.parquet")
df_clean = df_clean.merge(mapeo_template, on="Job Id", how="left")

print(f"Plantillas: {len(plantillas):,}  |  Embeddings: {embeddings.shape}")
print(f"Ofertas completas (con template_id reconstruido): {len(df_clean):,}")


## 3. Credenciales de Pinecone

1. Crea una cuenta gratuita en [pinecone.io](https://www.pinecone.io/) (el plan *Starter* gratuito alcanza de sobra para 3,760 vectores de 384 dimensiones).
2. En el dashboard, genera una **API Key**.
3. En Colab, ve al ícono de llave 🔑 en la barra lateral izquierda ("Secrets"), agrega un secreto llamado `PINECONE_API_KEY` con ese valor, y activa el acceso del notebook a ese secreto.

No pegues la API key directamente en una celda — así se mantiene fuera del notebook y de cualquier repositorio donde lo suban.

In [ ]:
from google.colab import userdata
from pinecone import Pinecone, ServerlessSpec

PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")
pc = Pinecone(api_key=PINECONE_API_KEY)
print("Cliente de Pinecone inicializado.")


## 4. Crear (o conectar a) el índice

Se crea un índice *serverless* — la modalidad que cubre el plan gratuito de Pinecone. La celda es segura de re-ejecutar: si el índice ya existe (por ejemplo, en una sesión anterior), no lo vuelve a crear, solo se conecta.

**Nota:** la combinación `cloud`/`region` disponible en el plan gratuito puede cambiar; si `us-east-1` en AWS da error de región no disponible, revisa en el dashboard de Pinecone qué región gratuita está habilitada para tu cuenta y ajusta el `ServerlessSpec`.

In [ ]:
NOMBRE_INDICE = "ofertas-laborales"
DIMENSION = embeddings.shape[1]  # 384 para all-MiniLM-L6-v2

indices_existentes = [i["name"] for i in pc.list_indexes()]

if NOMBRE_INDICE not in indices_existentes:
    print(f"Creando índice '{NOMBRE_INDICE}'...")
    pc.create_index(
        name=NOMBRE_INDICE,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    # Espera a que el índice quede listo antes de usarlo
    while not pc.describe_index(NOMBRE_INDICE).status["ready"]:
        time.sleep(1)
else:
    print(f"El índice '{NOMBRE_INDICE}' ya existe, se reutiliza.")

pinecone_index = pc.Index(NOMBRE_INDICE)
print(pinecone_index.describe_index_stats())


## 5. Subir (upsert) las plantillas al índice

Se sube cada plantilla como un vector con:
- **id**: el `template_id`, convertido a texto (Pinecone requiere IDs tipo string).
- **values**: el embedding de 384 dimensiones.
- **metadata**: `Job Title` y `Role`, para poder inspeccionar resultados directamente desde Pinecone sin tener que volver a cruzar con `df_clean` (aunque para el filtrado por país/salario/modalidad igual se usará `df_clean`, ver sección 7).

Se sube en lotes de 100 (límite recomendado por Pinecone por request).

In [ ]:
vectores = [
    {
        "id": str(fila["template_id"]),
        "values": embeddings[i].tolist(),
        "metadata": {"Job Title": fila["Job Title"], "Role": fila["Role"]},
    }
    for i, fila in plantillas.iterrows()
]

TAMANO_LOTE = 100
for inicio_lote in range(0, len(vectores), TAMANO_LOTE):
    lote = vectores[inicio_lote:inicio_lote + TAMANO_LOTE]
    pinecone_index.upsert(vectors=lote)

print(f"{len(vectores):,} plantillas subidas a Pinecone.")
print(pinecone_index.describe_index_stats())


## 6. Cargar el modelo de embeddings (para codificar consultas)

Los embeddings de las plantillas ya están calculados y subidos — el modelo solo hace falta ahora para convertir **la consulta del usuario** en un vector, igual que en el notebook 03.

In [ ]:
from sentence_transformers import SentenceTransformer

MODELO_EMBEDDINGS = "all-MiniLM-L6-v2"  # el elegido tras la comparación en 03_Embeddings_BusquedaVectorial.ipynb

modelo = SentenceTransformer(MODELO_EMBEDDINGS)
print(f"Modelo cargado: {MODELO_EMBEDDINGS}")


## 7. Búsqueda con Pinecone + expansión y filtros sobre `df_clean`

Misma lógica de `buscar_ofertas()` del notebook 03 (buscar plantillas relevantes → expandir a ofertas reales → filtrar por atributos estructurados), cambiando únicamente el motor de búsqueda vectorial: FAISS por Pinecone.

In [ ]:
def buscar_ofertas_pinecone(consulta: str, k_plantillas: int = 5, max_resultados: int = 20, filtros: dict | None = None):
    vector_consulta = modelo.encode(consulta, normalize_embeddings=True, convert_to_numpy=True).tolist()

    respuesta = pinecone_index.query(
        vector=vector_consulta,
        top_k=k_plantillas,
        include_metadata=False,  # no hace falta: los metadatos definitivos salen de df_clean
    )

    template_ids_relevantes = [int(match["id"]) for match in respuesta["matches"]]
    similitud_por_template = {int(match["id"]): match["score"] for match in respuesta["matches"]}

    resultados = df_clean[df_clean["template_id"].isin(template_ids_relevantes)].copy()
    resultados["similitud"] = resultados["template_id"].map(similitud_por_template)

    if filtros:
        for columna, valor in filtros.items():
            resultados = resultados[resultados[columna] == valor]

    resultados = resultados.sort_values("similitud", ascending=False)
    return resultados[["Job Title", "Role", "Country", "Work Type", "Salary Range", "similitud"]].head(max_resultados)

# Ejemplo
buscar_ofertas_pinecone("python developer with machine learning and NLP experience", k_plantillas=5)


In [ ]:
# Ejemplo con filtro estructurado
buscar_ofertas_pinecone(
    "python developer with machine learning and NLP experience",
    k_plantillas=5,
    filtros={"Work Type": "Full-Time"},
)


## 8. Comparación de latencia: FAISS (en memoria) vs. Pinecone (red)

Pinecone implica una llamada de red (HTTP) por cada consulta, mientras que FAISS busca en memoria local — es esperable que Pinecone sea más lento en milisegundos absolutos, aunque gane en persistencia y en que múltiples servicios (como la futura API) puedan consultarlo sin necesidad de tener el índice cargado en su propia RAM.

Esta celda asume que ya reconstruiste `indice_faiss` (sección 2.4 del notebook 03) en la sesión actual; si no, sáltala.

In [ ]:
def comparar_latencia_faiss_vs_pinecone(consulta: str, k: int = 5, repeticiones: int = 5):
    tiempos_faiss = []
    tiempos_pinecone = []

    for _ in range(repeticiones):
        vector_consulta = modelo.encode([consulta], normalize_embeddings=True, convert_to_numpy=True)

        inicio = time.time()
        indice_faiss.search(vector_consulta, k)
        tiempos_faiss.append(time.time() - inicio)

        inicio = time.time()
        pinecone_index.query(vector=vector_consulta[0].tolist(), top_k=k)
        tiempos_pinecone.append(time.time() - inicio)

    print(f"Consulta: \"{consulta}\"  (promedio de {repeticiones} repeticiones)")
    print(f"  FAISS (memoria):    {np.mean(tiempos_faiss) * 1000:.1f} ms")
    print(f"  Pinecone (red):     {np.mean(tiempos_pinecone) * 1000:.1f} ms")

comparar_latencia_faiss_vs_pinecone("python developer with machine learning and NLP experience")


## 9. Próximos pasos

1. Validar que los resultados de `buscar_ofertas_pinecone()` coinciden razonablemente con los de `buscar_ofertas()` (FAISS) del notebook 03 — deberían, ya que ambos usan la misma métrica (coseno) sobre los mismos vectores.
2. Con esta base funcionando, la futura **API** (Objetivo específico 4) puede conectarse directamente a este mismo índice de Pinecone en vez de tener que reconstruir un índice FAISS en memoria cada vez que arranque el servidor.
3. Evaluar si el plan gratuito de Pinecone es suficiente para el resto del proyecto, o si conviene migrar a una alternativa self-hosted (Qdrant/Chroma) en el VPS de Railway ya presupuestado — sobre todo si en algún momento se decide indexar las 1.6M ofertas individuales (Opción B) en vez de solo las plantillas.
4. Sigue pendiente el filtrado por **rango** (`experiencia_min/max`, `salario_min/max`) — por ahora `buscar_ofertas_pinecone()` solo filtra por igualdad exacta, igual que la versión de FAISS.